## Run this notebook

You can launch this notebook in VEDA JupyterHub by clicking the link below.

[Launch in VEDA JupyterHub (requires access)](https://hub.openveda.cloud/hub/user-redirect/git-pull?repo=https://github.com/NASA-IMPACT/veda-docs&urlpath=lab/tree/veda-docs/user-guide/notebooks/quickstarts/open-and-plot.ipynb&branch=main) 

<details><summary>Learn more</summary>
    
### Inside the Hub

This notebook was written on the VEDA JupyterHub and as such is designed to be run on a jupyterhub which is associated with an AWS IAM role which has been granted permissions to the VEDA data store via its bucket policy. The instance used provided 16GB of RAM. 

See (VEDA Analytics JupyterHub Access)[https://nasa-impact.github.io/veda-docs/veda-jh-access.html] for information about how to gain access.

### Outside the Hub

The data is in a protected bucket. Please request access by emailng aimee@developmentseed.org or alexandra@developmentseed.org and providing your affiliation, interest in or expected use of the dataset and an AWS IAM role or user Amazon Resource Name (ARN). The team will help you configure the cognito client.

You should then run:

```
%run -i 'cognito_login.py'
```
    
</details>

## Approach

   1. Use `pystac_client` to open the STAC catalog and retrieve the items in the collection
   2. Open the collection with `xarray` and `odc-stac`
   3. Plot the data using `hvplot`

## About the data
    
CDC's Social Vulnerability Index (SVI) uses 15 variables at the census tract level. The data comes from the U.S. decennial census for the years 2000 & 2010, and the American Community Survey (ACS) for the years 2014, 2016, and 2018. It is a hierarchical additive index (Tate, 2013), with the component elements of CDC’s SVI including the following for 4 themes: Socioeconomic Status, Household Composition & Disability, Minority Status & Language, and Housing Type & Transportation.

SVI indicates the relative vulnerability of every U.S. Census tract–subdivisions of counties for which the Census collects statistical data. SVI ranks the tracts on 15 social factors, including unemployment, minority status, and disability, and further groups them into four related themes. Thus, each tract receives a ranking for each Census variable and for each of the four themes, as well as an overall ranking.

### Scientific research

The SVI Overall Score provides the overall, summed social vulnerability score for a given tract. The Overall Score SVI Grid is part of the U.S. Census Grids collection, and displays the Center for Disease Control & Prevention (CDC) SVI score. Funding for the final development, processing and dissemination of this data set by the Socioeconomic Data and Applications Center (SEDAC) was provided under the U.S. National Aeronautics and Space Administration (NASA)¹.

The Overall SVI Score describes the vulnerability in a given county tract based on the combined percentile ranking of the four SVI scores (Socioeconomic Status, Household Composition & Disability, Minority Status & Language, and Housing Type & Transportation). The summed percentile ranking from the four themes is ordered, and then used to calculate an overall percentile ranking, ranging from 0 (less vulnerable) to 1 (more vulnerable)². Tracts with higher Overall SVI Scores typically rank high in other SVI domains, and reveal communities that may require extra support, resources, and preventative care in order to better prepare for and manage emergency situations.

### Interpreting the data

The Overall SVI Score describes the vulnerability in a given county tract based on the combined percentile ranking of the four SVI scores (Socioeconomic Status, Household Composition & Disability, Minority Status & Language, and Housing Type & Transportation). The summed percentile ranking from the four themes is ordered, and then used to calculate an overall percentile ranking, ranging from 0 (less vulnerable) to 1 (more vulnerable)². Tracts with higher Overall SVI Scores typically rank high in other SVI domains, and reveal communities that may require extra support, resources, and preventative care in order to better prepare for and manage emergency situations.

### Credits

* Center for International Earth Science Information Network, (CIESIN), Columbia University. 2021. Documentation for the U.S. Social Vulnerability Index Grids. Palisades, NY: NASA Socioeconomic Data and Applications Center (SEDAC). https://doi.org/10.7927/fjr9-a973. Accessed 13 May 2022.
* Centers for Disease Control and Prevention/ Agency for Toxic Substances and Disease Registry/ Geospatial Research, Analysis, and Services Program. CDC/ATSDR Social Vulnerability Index Database. https://www.atsdr.cdc.gov/placeandhealth/svi/documentation/pdf/SVI2018Documentation_01192022_1.pdf


In [1]:
import hvplot.xarray  # noqa
import odc.stac
from pystac_client import Client

## Declare your collection of interest

You can discover available collections the following ways:

* Programmatically: see example in the `list-collections.ipynb` notebook
* JSON API: https://openveda.cloud/api/stac/collections
* STAC Browser: https://openveda.cloud

In [2]:
STAC_API_URL = "https://openveda.cloud/api/stac/"
collection = "social-vulnerability-index-overall-nopop"

## Find items in collection

Use `pystac_client` to search the STAC collection.

In [3]:
catalog = Client.open(STAC_API_URL)
search = catalog.search(collections=[collection])

item_collection = search.item_collection()
print(f"Found {len(item_collection)} items")

Found 5 items


## Read data

Read in data using `xarray` using a combination of `xpystac`, `odc-stac`, and `rasterio`.

In [4]:
%%time
da = odc.stac.load(item_collection, chunks={"latitude": "auto", "longitude": "auto"})

# Grab the data band and mask it
data = da["cog_default"]
data = data.where(data != data.nodata)
data

CPU times: user 241 ms, sys: 30.8 ms, total: 272 ms
Wall time: 277 ms


<xarray.DataArray 'cog_default' (time: 5, latitude: 6297, longitude: 13353)> Size: 2GB
dask.array<where, shape=(5, 6297, 13353), dtype=float32, chunksize=(1, 5792, 5792), chunktype=numpy.ndarray>
Coordinates:
  * time         (time) datetime64[ns] 40B 2000-01-01 2010-01-01 ... 2018-01-01
  * latitude     (latitude) float64 50kB 71.38 71.37 71.36 ... 18.93 18.92 18.91
  * longitude    (longitude) float64 107kB -178.2 -178.2 ... -66.97 -66.96
    spatial_ref  int32 4B 4326
Attributes:
    nodata:   -3.3999999521443642e+38

There are 5 items representing the 5 years of data in the collection (2000, 2010, 2014, 2016, and 2018).

## Plot data

Plot data using `hvplot`. By using `rasterize=True` we tell `hvplot` to use `datashader` behind the scenes to make the plot render more quickly and re-render on zoom.

In [5]:
%%time
data.compute().hvplot(
    x="longitude",
    y="latitude",
    rasterize=True,
    clim=(0, 1),
    coastline=True,
    cmap="viridis",
    widget_location="bottom",
)

CPU times: user 8.57 s, sys: 3.75 s, total: 12.3 s
Wall time: 7.64 s


Column
    [0] HoloViews(DynamicMap, sizing_mode='fixed', widget_location='bottom')
    [1] WidgetBox(align=('center', 'end'))
        [0] DiscreteSlider(name='time', options={'2000-01-01 00:00:00': np...}, value=np.datetime64('2000-01-01T...)